# Credit Risk Analysis
## Objetivo
Predecir la probabilidad de que un cliente caiga en morosidad grave 
en los próximos 2 años, utilizando variables financieras y demográficas.

## Dataset
- Fuente: Give Me Some Credit — Kaggle
- Tamaño: 150,000 registros
- Variable objetivo: SeriousDlqin2yrs (1 = moroso, 0 = no moroso)

In [3]:
#Importamos librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


In [5]:
#Cargamos la data
df = pd.read_csv('../data/cs-training.csv')
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [10]:
#Inspeccion Inicial:
#registros y columnas
print(df.shape)

#columnas con tipo de datos para identificar nulos tambien
print(df.info())

#Analisis estadistico inicial
df.describe()


(150000, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,75000.500000,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,43301.414527,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37500.750000,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,75000.500000,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,112500.250000,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,150000.000000,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


Se puede identificar que el dataset presenta 150 000 registros y 12 columnas.
Ademas se identifica valores vacios en las columnas MonthlyIncome y NumberOfDependents
Ademas de que en el analisis estadistico se ve que hay personas con 0 edad y de 109 aunque este ultimo quiza sea posible pero muy raramente, lo cual resulta imposible e ilogico ya que un recien nacido no podria sacar un credito, ademas de personas con ingresos mensuales de 0, que banco podria darle credito a una persona que no tiene ingresos.

In [22]:
#Identificamos nulos en MonthlyIncome 
print('Nulos en MonthlyIncome:', df['MonthlyIncome'].isnull().sum())
print('Porcentaje de nulos en MonthlyIncome en base al total:',df['MonthlyIncome'].isnull().sum()/len(df)*100)

#Identificamos nulos en NumberOfDependents 
print('Nulos en NumberOfDependents:',df['NumberOfDependents'].isnull().sum())
print('Porcentaje de nulos en NumberOfDependents en base al total:',df['NumberOfDependents'].isnull().sum()/len(df)*100)

Nulos en MonthlyIncome: 29731
Porcentaje de nulos en MonthlyIncome en base al total: 19.820666666666668
Nulos en NumberOfDependents: 3924
Porcentaje de nulos en NumberOfDependents en base al total: 2.616


In [25]:
#Vamos a verificar si la media y la mediana de monthlyincome es muy diferente
print('==MonthlyIncome==')
print('Media:',df['MonthlyIncome'].mean())
print('Mediana:',df['MonthlyIncome'].median())

print('\n==NumberOfDependents==')
print('Media', df['NumberOfDependents'].mean())
print('Mediana', df['NumberOfDependents'].median())

==MonthlyIncome==
Media: 6670.221237392844
Mediana: 5400.0

==NumberOfDependents==
Media 0.7572222678605657
Mediana 0.0


In [27]:
#Imputamos los nulos, es decir reemplazamos los nulos por un valor
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

In [28]:
print(df.isnull().sum())

Unnamed: 0                              0
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64


In [31]:
#Vamos a eliminar la columna Unnamed: 0 
df=df.drop(columns='Unnamed: 0')


In [33]:
print(df.shape)

(150000, 11)


In [34]:
print(df['age'].describe())

count    150000.000000
mean         52.295207
std          14.771866
min           0.000000
25%          41.000000
50%          52.000000
75%          63.000000
max         109.000000
Name: age, dtype: float64


In [42]:
print('Edad <18: ', (df['age']<18).sum())
print('Edad >100: ', (df['age']>100).sum())

Edad <18:  1
Edad >100:  13


In [44]:
#Eliminamos los registros con edades menores a 18 y mayores a 100

df = df[(df['age']>=18) & (df['age']<=100)]

In [45]:
print(df.shape)

(149986, 11)


## Decisiones de limpieza

- **Nulos en MonthlyIncome (19.8%):** Imputados con mediana ($5,400) 
  por distribución sesgada hacia la derecha.
- **Nulos en NumberOfDependents (2.6%):** Imputados con mediana (0) 
  por ser el valor más frecuente.
- **Columna Unnamed: 0:** Eliminada por ser índice sin valor analítico.
- **Registros con edad < 18 o > 100:** 14 filas eliminadas por ser 
  imposibles en contexto de crédito bancario.